# Day 1 - Selenium Edition

Drop-in replacement for `day1.ipynb` that uses **Selenium** instead of `requests` to scrape websites.

### Why Selenium?
The original scraper fails on JavaScript-rendered pages (e.g. `openai.com`, `anthropic.com`, `huggingface.co`). Selenium launches a real Chrome browser behind the scenes, waits for JS to execute, and extracts the rendered HTML — so those sites work.

### Setup
```bash
pip install selenium webdriver-manager beautifulsoup4
```
No manual ChromeDriver download needed — `webdriver-manager` handles it automatically.

In [1]:
import os
from dotenv import load_dotenv
from scraper_selenium import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please check your .env file")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters - please remove them")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


## System Prompt — Tech Intelligence Analyst

Instead of a simple summarizer, we use a more detailed analyst prompt that extracts structured intelligence from any website: what the company does, key products, recent news, and a one-line verdict.

In [3]:
system_prompt = """
You are a sharp technology intelligence analyst. When given the scraped text of a website, produce a structured report with these sections:

## What They Do
One or two sentences on the company/site's core purpose.

## Key Products or Services
Bullet list of the main offerings mentioned.

## Latest News or Announcements
Bullet list of any recent updates, releases, or blog posts. Write "None found" if the page has none.

## Analyst Verdict
A single sentence — your blunt, honest take on what this company is trying to be.

Respond in markdown. Be concise and direct — no filler, no hype.
"""

In [4]:
user_prompt_prefix = """
Here is the scraped content of a website. Produce your structured intelligence report.

"""

In [5]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website.get_contents()}
    ]

In [6]:
openai = OpenAI()

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    print(f"Scraping: {url}")
    summary = summarize(url)
    display(Markdown(summary))

## Difficult JS-heavy sites — these fail with the original requests-based scraper

In [7]:
# OpenAI — heavy React app, blank page without JS execution
display_summary("https://openai.com")

Scraping: https://openai.com


## What They Do
OpenAI develops advanced artificial intelligence models and tools designed to enhance human workflows, research, and business applications.

## Key Products or Services
- ChatGPT conversational AI platform
- Codex coding assistant integrated into multiple roles and workflows
- ChatGPT Enterprise for business applications
- AI agents for automation and operational improvements
- Specialized AI models like GPT-Rosalind for life sciences research
- Security tools such as Daybreak for organizational protection
- Features improving AI safety and content provenance

## Latest News or Announcements
- Confidential draft S-1 submitted to SEC (Jun 8, 2026)
- Public plan focused on benefiting everyone (Jun 8, 2026)
- Advancements in content provenance for AI transparency (May 19, 2026)
- New personal finance experience within ChatGPT (May 15, 2026)
- Enhancements in handling sensitive conversations and safety features (May 7 - May 14, 2026)
- Research breakthroughs including disproving a conjecture in discrete geometry (May 20, 2026)
- Introduction of GPT-Rosalind for life sciences (Apr 16, 2026)
- Business case studies featuring AI-driven automation and enterprise adoption

## Analyst Verdict
OpenAI is positioning itself as the leading general-purpose AI platform provider, aiming to embed intelligent automation and advanced research capabilities across industries.

In [8]:
# Anthropic — JS-rendered, returns almost nothing without a real browser
display_summary("https://anthropic.com")

Scraping: https://anthropic.com


## What They Do
Anthropic is an AI research and product company focused on developing safe and responsible artificial intelligence technologies.

## Key Products or Services
- Claude (AI assistant/model)
- Claude Code (coding assistant)
- Claude Cowork (collaborative AI tools)
- Claude Design
- Claude Security
- Claude Platform
- Various AI models: Mythos, Fable, Opus, Sonnet, Haiku
- Developer resources: tutorials, docs, use cases, Anthropic Academy

## Latest News or Announcements
- Expansion of Project Glasswing to ~150 new organizations
- Release of a major AI impact study involving 81,000 people
- Continued emphasis on AI safety, transparency, and responsible scaling policies

## Analyst Verdict
Anthropic aims to be a leader in creating AI technologies with a strong focus on safety and ethical impact rather than just commercial dominance.

In [9]:
# HuggingFace — dynamic model cards and trending sections need JS
display_summary("https://huggingface.co")

Scraping: https://huggingface.co


## What They Do
Hugging Face is an AI community platform focused on collaboration around machine learning models, datasets, and applications.

## Key Products or Services
- Hosting and browsing 2 million+ machine learning models  
- Access to 500k+ datasets  
- Spaces for running and sharing AI applications and agents  
- Enterprise solutions including Hugging Face PRO, enterprise support, inference endpoints, and storage buckets  
- Community features like forums, Discord, blogs, and collaborative tools  
- Learning resources including daily papers and documentation  

## Latest News or Announcements
- Frequent updates to popular models such as zai-org/GLM-5.2 (updated 9 hours ago), yuxinlu1/gemma-4-12B (updated 3-4 days ago), and MiniMaxAI/MiniMax-M3 (updated 11 hours ago)  
- Updates to datasets like Glint-Research/Fable-5-traces (4 days ago) and armand0e/claude-fable-5-claude-code (3 days ago)  
- Active deployment of featured AI Spaces and agents such as LTX 2.3 Finetuned I2V, Gemma 4 WebGPU Kernels, and OpenMythos cybersecurity agent  

## Analyst Verdict
Hugging Face aims to be the central collaborative hub and marketplace for open machine learning models, datasets, and AI applications across community and enterprise users.

In [10]:
# GitHub Trending — entirely JS-rendered, empty with requests
display_summary("https://github.com/trending")

Scraping: https://github.com/trending


## What They Do
GitHub is a platform for software development and version control that enables collaboration, code hosting, and automation throughout the development lifecycle.

## Key Products or Services
- GitHub Copilot (AI-assisted code creation)
- GitHub Actions (workflow automation)
- Codespaces (instant development environments)
- GitHub Advanced Security (vulnerability detection and code security)
- Secret protection (preventing credential leaks)
- Issue tracking and project planning tools
- Code review and collaboration features
- Enterprise platform with AI-powered developer tools
- Additional services including premium support and GitHub Sponsors for funding open source developers

## Latest News or Announcements
- MCP Registry (new feature to integrate external tools)
- Continuous updates to GitHub Copilot and Advanced Security (implied from menu context)
- No explicit recent announcements or dated news found on the page

## Analyst Verdict
GitHub aims to be the comprehensive platform for collaborative software development enhanced with AI and security tools to streamline the entire coding and deployment process.

In [ ]:
# TechCrunch — news site with heavy ad/tracking scripts that block basic scrapers
display_summary("https://techcrunch.com")